# Build and load xMAE

In [ ]:
# !pip install torchinfo 
import torch
import torchinfo
from utils.model_arch.xmae import *
# from utils_xmae import *
import warnings
warnings.filterwarnings("ignore")
from collections import OrderedDict
from typing import Optional

def build_and_load_xmae(filepath: Optional[str] = None):

    cfg = {
        'sampling_freq': 100,
        'seg_len': 10, 
        'source': 'ppg+ecg',
        'model_params': {
            'latent_dim': 256, 
            'd_model': 256,
            'nhead': 8,
            'depth_ecg': 1,
            'depth_ppg': 2,
            'depth_bridge': 1,
            'stem_ch': 32,
            'dropout': 0.1,
            'use_cross_bridge': True,
            'patch_len': 40,
        },
    }

    base_model = build_model_from_cfg(cfg)
    if filepath is None:
        return base_model

    checkpoints = torch.load(filepath, map_location="cuda" if torch.cuda.is_available() else "cpu")
    
    new_state = OrderedDict(
        (k[len('model.'):], v) for k, v in checkpoints.items() if k.startswith('model.') 
    )
    base_model.load_state_dict(new_state, strict=True)
    
    return base_model
   

BACKBONE_CKPT = "xmae_weights_permute.pth"
xmae_backbone = build_and_load_xmae(BACKBONE_CKPT)

B = 2
L = 1000
torchinfo.summary(xmae_backbone, input_size=(B, L))



Layer (type:depth-idx)                                  Output Shape              Param #
xMAE                                                    [2, 256]                  3,636,485
├─SinglePath: 1-1                                       --                        550,145
│    └─UNetHalfInceptionStem1D: 2-1                     [2, 32, 1000]             --
│    │    └─Sequential: 3-1                             [2, 32, 1000]             7,716
│    │    └─ConvBNAct: 3-2                              [2, 64, 500]              6,272
│    │    └─Sequential: 3-3                             [2, 64, 500]              49,920
│    │    └─ConvBNAct: 3-4                              [2, 128, 250]             24,832
│    │    └─Sequential: 3-5                             [2, 128, 250]             198,656
│    │    └─Conv1d: 3-6                                 [2, 64, 500]              4,096
│    │    └─Conv1d: 3-7                                 [2, 64, 250]              8,192
│    │    └─ConvBNAct: 

# Utilize PPG encoder for linear-probing or finetuning

In [2]:

# ----------------------------
# model wrapper
# ----------------------------
class cls_wrapper(torch.nn.Module):

    def __init__(self, 
                 backbone: torch.nn.Module,
                 device: torch.device,
                 is_linear_probe: bool = True, 
                 n_class: int = 2, # binary classfication
                 d_xmae: int = 256, # latent dim of xMAE
                 ):
        super().__init__()
        self.backbone = backbone
        self.device = device
        
        for param in self.backbone.parameters():
            param.requires_grad = False if is_linear_probe else True 

        self.cls_head = nn.Linear(d_xmae, n_class)

 
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        feats = self.backbone(
            ppg=x.to(self.device), ecg=None,
            ids_keep_ppg=None, ids_restore_ppg=None,
            ids_keep_ecg=None, ids_restore_ecg=None
        )
        latents = feats['ppg_embedding'] 
        return latents, self.cls_head(latents) # [B, d_xmae], [B, n_class]
       
       
def min_max_norm(signal):
    min_val = signal.min(dim=0, keepdim=True).values
    max_val = signal.max(dim=0, keepdim=True).values
    return 2 * (signal - min_val) / (max_val - min_val) - 1
 
# ----------------------------
# Hyperparameters
# ----------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
B = 2
L = 1000


# ----------------------------
# probe model input
# ----------------------------
probe_model = cls_wrapper(xmae_backbone, device=device)


# ----------------------------
# signals input and forward pass
# ----------------------------
ppg_signal1 = min_max_norm(torch.rand(B, L))
ppg_embed, logits = probe_model(ppg_signal1)
print(ppg_embed.shape, logits.shape) 


torch.Size([2, 256]) torch.Size([2, 2])


# Utilize xMAE for ECG reconstruction

In [3]:

   
def make_trailing_ecg_mask(
    B: int, N: int, ratio: float, device: torch.device
) -> torch.Tensor:
    """
    Return patch_mask (B, N) with 1=masked, 0=keep.
    Trailing mask: first Nk visible, last k masked.
    """
    k = int(round(ratio * N))
    k = max(0, min(N, k))
    m = torch.zeros(B, N, device=device)
    if k > 0:
        m[:, N - k:] = 1.0
    return m


def ids_from_patch_mask(patch_mask: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor, int]:
    """
    From patch_mask (B,N) with 1=masked, 0=keep:
      -> ids_keep (B,Nk), ids_restore (B,N), Nk (int)
    """
    B, N = patch_mask.shape
    device = patch_mask.device
    base_idx = torch.arange(N, device=device).view(1, N).expand(B, N)
    keep = patch_mask <= 0.0
    Nk_each = keep.sum(dim=1)
    assert int(Nk_each.min().item()) == int(Nk_each.max().item()), \
        "All items in the batch must have the same number of kept patches."
    Nk = int(Nk_each[0].item())
    scores = base_idx + (patch_mask > 0.0).float() * N  # keeps rank lower
    ids_sorted = torch.argsort(scores, dim=1)           # (B,N)
    ids_keep = ids_sorted[:, :Nk]                       # (B,Nk)
    ids_restore = torch.argsort(ids_sorted, dim=1)      # (B,N)
    return ids_keep.long(), ids_restore.long(), Nk


# ----------------------------
# Hyperparameters
# ----------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
B = 1
L = 1000
PATCH_SZ = 40
n_patches = L // PATCH_SZ
ecg_mask_ratio = 0.9


# ----------------------------
# signals input
# ----------------------------
ppg_signal2 = min_max_norm(torch.rand(B, L))
ecg_signal2 = min_max_norm(torch.rand(B, L))
ppg = ppg_signal2.to(device)
ecg = ppg_signal2.to(device)
  
# ----------------------------
# Build ECG mask (visible first, masked tail)
# ----------------------------
pm_ecg = make_trailing_ecg_mask(B, n_patches, ecg_mask_ratio, device)
ids_keep_ecg, ids_restore_ecg, Nk_ecg = ids_from_patch_mask(pm_ecg)

# ----------------------------
# Forward pass
# ----------------------------
outputs = xmae_backbone(
    ppg=ppg,
    ecg=ecg,
    ids_keep_ppg=None, # keep PPG fully visible by passing None
    ids_restore_ppg=None,
    ids_keep_ecg=ids_keep_ecg,
    ids_restore_ecg=ids_restore_ecg,
)

# Get reconstructed ecg
ecg_rec = outputs.get("ecg_reconstructed")
print(ecg_rec.shape)
    

torch.Size([1, 1, 1000])
